In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import hydra

from genpp import BASE_DIR
from genpp.configs import register_resolvers

In [3]:
register_resolvers()

# Test if Model works in general

In [4]:
with hydra.initialize_config_dir(version_base=None, config_dir=str(BASE_DIR / "configs")):
    cfg = hydra.compose(config_name="base_chen")

In [5]:
datamodule = hydra.utils.instantiate(cfg.data.module)
datamodule.prepare_data()
datamodule.setup("fit")
dataloader = datamodule.train_dataloader()

Configuration hash: 05a2d862df574c3ec09a73a2516597b76945436332ad35785c3a5fe80dbc1c8c
Cached tensor data found. Verifying configuration...
Using cached tensor data from /Users/moritzfeik/Developer/GENPP/src/genpp/data/weatherbench2/cache/tensor_05a2d862df574c3ec09a73a2516597b76945436332ad35785c3a5fe80dbc1c8c.pt.


In [6]:
cfg.model.n_samples_train = 16  # for testing purposes only

model = hydra.utils.instantiate(
    cfg.model, rescaler=datamodule.y_reverseModules if cfg.model.use_rescaler else None
)

In [8]:
trainer = hydra.utils.instantiate(
    cfg.trainer,
    logger=None,
    detect_anomaly=True,
    accelerator="cpu",
    max_epochs=50,
    overfit_batches=10,
    check_val_every_n_epoch=5,
)
trainer.fit(model, datamodule=datamodule)

You have turned on `Trainer(detect_anomaly=True)`. This will significantly slow down compute speed and is recommended only for model debugging.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Users/moritzfeik/Developer/GENPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
/Users/moritzfeik/Developer/GENPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoa

/Users/moritzfeik/Developer/GENPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:252: You requested to overfit but enabled train dataloader shuffling. We are turning off the train dataloader shuffling for you.


Epoch 43:  80%|████████  | 8/10 [01:14<00:18,  0.11it/s, v_num=3, train_loss=110.0, val_loss_var_0=100.0, val_loss_var_1=88.00, val_loss=123.0] 


Detected KeyboardInterrupt, attempting graceful shutdown ...


: 